In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

# ============================================================
# Configuration
# ============================================================

SOURCE_PATH = (
    "abfss://source@stworkplaceanalyticsdev.dfs.core.windows.net/customers/"
)

BRONZE_TABLE = "dev_catalog.bronze.customer"

SCHEMA_LOCATION = (
    "abfss://bronze@stworkplaceanalyticsdev.dfs.core.windows.net/"
    "_schemas/customer/"
)

CHECKPOINT_LOCATION = (
    "abfss://bronze@stworkplaceanalyticsdev.dfs.core.windows.net/"
    "_checkpoints/customer/"
)

LOG_TABLE = "dev_catalog.control.pipeline_run_log"

PIPELINE_NAME = "customer_etl_job"
TASK_NAME = "bronze_customer"


# ============================================================
# Generate unique run ID
# ============================================================

run_id = str(
    spark.sql("SELECT uuid()").first()[0]
)

start_timestamp = spark.sql(
    "SELECT current_timestamp()"
).first()[0]


print(f"Starting Bronze ingestion. Run ID: {run_id}")


# ============================================================
# Explicit source schema
# ============================================================

customer_schema = StructType([
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Phone", StringType(), True),
    StructField("Modified_Date", TimestampType(), True)
])


# ============================================================
# Pipeline log schema
# ============================================================

log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("task_name", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("records_processed", LongType(), True),
    StructField("records_rejected", LongType(), True),
    StructField("error_message", StringType(), True)
])


# ============================================================
# Bronze Processing
# ============================================================

try:

    print("Starting Bronze Auto Loader ingestion...")

    # ========================================================
    # Read source using Auto Loader
    # ========================================================

    df = (
        spark.readStream
        .format("cloudFiles")
        .option(
            "cloudFiles.format",
            "csv"
        )
        .option(
            "cloudFiles.schemaLocation",
            SCHEMA_LOCATION
        )
        .option(
            "header",
            "true"
        )
        .schema(customer_schema)
        .load(SOURCE_PATH)
    )


    # ========================================================
    # Add audit columns
    # ========================================================

    bronze_df = (
        df
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
        .withColumn(
            "_run_id",
            F.lit(run_id)
        )
    )


    # ========================================================
    # Write Bronze
    # ========================================================

    query = (
        bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            CHECKPOINT_LOCATION
        )
        .trigger(
            availableNow=True
        )
        .toTable(BRONZE_TABLE)
    )

    query.awaitTermination()


    # ========================================================
    # Get records processed for this run
    #
    # Because Auto Loader is incremental, only records
    # written during this execution have this run_id.
    # ========================================================

    records_processed = spark.sql(
        f"""
        SELECT COUNT(*)
        FROM {BRONZE_TABLE}
        WHERE _run_id = '{run_id}'
        """
    ).first()[0]


    end_timestamp = spark.sql(
        "SELECT current_timestamp()"
    ).first()[0]


    # ========================================================
    # Write SUCCESS log
    # ========================================================

    log_df = spark.createDataFrame(
        [(
            run_id,
            PIPELINE_NAME,
            TASK_NAME,
            start_timestamp,
            end_timestamp,
            "SUCCESS",
            records_processed,
            0,
            None
        )],
        schema=log_schema
    )


    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )


    print(
        f"Bronze ingestion completed successfully. "
        f"Records processed: {records_processed}"
    )


# ============================================================
# Failure handling
# ============================================================

except Exception as e:

    end_timestamp = spark.sql(
        "SELECT current_timestamp()"
    ).first()[0]

    error_message = str(e)


    # ========================================================
    # Write FAILED log
    # ========================================================

    log_df = spark.createDataFrame(
        [(
            run_id,
            PIPELINE_NAME,
            TASK_NAME,
            start_timestamp,
            end_timestamp,
            "FAILED",
            0,
            0,
            error_message
        )],
        schema=log_schema
    )


    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )


    print(
        f"Bronze ingestion failed: {error_message}"
    )


    raise